# 第 28 章：光谱分类案例

这个 notebook 把预处理、PCA、分类和错误案例复查串成一个完整的小型光谱项目：

- 从一组 SDSS/LAMOST 风格教学光谱矩阵开始
- 修补坏像元并做按谱归一化
- 用 PCA 观察主要结构
- 在 PCA 空间里做最近中心分类
- 回到错分样本，检查它更像数据问题还是边界对象


In [ ]:
from __future__ import annotations

import csv
import math
from pathlib import Path

DATA_PATH = Path("../../data/small/spectral_classification_case_demo.csv").resolve()

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = []
    for raw in csv.DictReader(handle):
        row = {
            "spectrum_id": raw["spectrum_id"],
            "object_class": raw["object_class"],
            "snr": float(raw["snr"]),
            "quality_flag": raw["quality_flag"],
        }
        for key, value in raw.items():
            if key.startswith("flux_"):
                row[key] = float(value)
        rows.append(row)

flux_names = [name for name in rows[0] if name.startswith("flux_")]
flux_names.sort()

print(f"Loaded {len(rows)} spectra from {DATA_PATH.name}")
print("flux columns:", flux_names)
class_counts = {}
quality_counts = {}
for row in rows:
    class_counts[row["object_class"]] = class_counts.get(row["object_class"], 0) + 1
    quality_counts[row["quality_flag"]] = quality_counts.get(row["quality_flag"], 0) + 1
print("class counts:", class_counts)
print("quality counts:", quality_counts)
print("first spectrum:", rows[0])


In [ ]:
def median(values):
    sorted_values = sorted(values)
    return sorted_values[len(sorted_values) // 2]

def repair_bad_pixels(row, flux_names):
    repaired = []
    for index, flux_name in enumerate(flux_names):
        if row[flux_name] >= 0.0:
            continue
        left_value = next(
            (row[flux_names[left_index]] for left_index in range(index - 1, -1, -1) if row[flux_names[left_index]] >= 0.0),
            None,
        )
        right_value = next(
            (row[flux_names[right_index]] for right_index in range(index + 1, len(flux_names)) if row[flux_names[right_index]] >= 0.0),
            None,
        )
        if left_value is None:
            replacement = right_value
        elif right_value is None:
            replacement = left_value
        else:
            replacement = 0.5 * (left_value + right_value)
        row[flux_name] = replacement
        repaired.append((flux_name, round(replacement, 3)))
    return repaired

for row in rows:
    row["repair_log"] = repair_bad_pixels(row, flux_names)
    flux_values = [row[flux_name] for flux_name in flux_names]
    row["normalization_median"] = median(flux_values)
    for flux_name in flux_names:
        row[flux_name + "_norm"] = row[flux_name] / row["normalization_median"]
    row["blue_red_slope"] = row["flux_3900_norm"] - row["flux_6800_norm"]
    row["balmer_index"] = (
        (row["flux_3900_norm"] + row["flux_4300_norm"]) / 2.0 - row["flux_4100_norm"]
        + (row["flux_4300_norm"] + row["flux_5175_norm"]) / 2.0 - row["flux_4861_norm"]
    )
    row["halpha_excess"] = row["flux_6563_norm"] - (row["flux_5175_norm"] + row["flux_6800_norm"]) / 2.0

repaired_rows = [(row["spectrum_id"], row["repair_log"]) for row in rows if row["repair_log"]]
print("repaired rows:", repaired_rows)
print("sample normalized fluxes:")
for row in rows[:4]:
    normalized_fluxes = [round(row[flux_name + "_norm"], 3) for flux_name in flux_names]
    print(row["spectrum_id"], row["object_class"], normalized_fluxes)

index_summary = {}
for row in rows:
    index_summary.setdefault(row["object_class"], {"blue_red_slope": [], "balmer_index": [], "halpha_excess": []})
    for key in ["blue_red_slope", "balmer_index", "halpha_excess"]:
        index_summary[row["object_class"]][key].append(row[key])

print("index ranges by class:")
for object_class, values in index_summary.items():
    print(
        object_class,
        {
            key: (round(min(series), 3), round(max(series), 3))
            for key, series in values.items()
        },
    )


In [ ]:
normalized_flux_names = [flux_name + "_norm" for flux_name in flux_names]
standardized_flux_names = [flux_name + "_std" for flux_name in normalized_flux_names]

feature_means = {}
feature_stds = {}
for feature_name in normalized_flux_names:
    values = [row[feature_name] for row in rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    feature_means[feature_name] = mean_value
    feature_stds[feature_name] = math.sqrt(variance) or 1.0

for row in rows:
    for feature_name in normalized_flux_names:
        row[feature_name + "_std"] = (row[feature_name] - feature_means[feature_name]) / feature_stds[feature_name]

def dot(left_vector, right_vector):
    return sum(left * right for left, right in zip(left_vector, right_vector))

def matvec(matrix, vector):
    return [sum(matrix[row_index][column_index] * vector[column_index] for column_index in range(len(vector))) for row_index in range(len(matrix))]

def vector_norm(vector):
    return math.sqrt(dot(vector, vector))

def power_iteration(matrix, steps=80):
    vector = [1.0] + [0.5] * (len(matrix) - 1)
    length = vector_norm(vector)
    vector = [value / length for value in vector]
    for _ in range(steps):
        next_vector = matvec(matrix, vector)
        length = vector_norm(next_vector)
        vector = [value / length for value in next_vector]
    eigenvalue = dot(vector, matvec(matrix, vector))
    return eigenvalue, vector

feature_matrix = [[row[feature_name] for feature_name in standardized_flux_names] for row in rows]
covariance_matrix = [
    [sum(vector[row_index] * vector[column_index] for vector in feature_matrix) / len(feature_matrix) for column_index in range(len(feature_matrix[0]))]
    for row_index in range(len(feature_matrix[0]))
]
total_variance = sum(covariance_matrix[index][index] for index in range(len(covariance_matrix)))

eigenvalue1, pc1_vector = power_iteration(covariance_matrix)
deflated_covariance = [
    [covariance_matrix[row_index][column_index] - eigenvalue1 * pc1_vector[row_index] * pc1_vector[column_index] for column_index in range(len(covariance_matrix))]
    for row_index in range(len(covariance_matrix))
]
eigenvalue2, pc2_vector = power_iteration(deflated_covariance)

for row in rows:
    standardized_vector = [row[feature_name] for feature_name in standardized_flux_names]
    row["pc1"] = dot(standardized_vector, pc1_vector)
    row["pc2"] = dot(standardized_vector, pc2_vector)

pc1_loadings = sorted(zip(flux_names, pc1_vector), key=lambda item: abs(item[1]), reverse=True)
pc2_loadings = sorted(zip(flux_names, pc2_vector), key=lambda item: abs(item[1]), reverse=True)
print("PCA explained variance ratios:", {"pc1": round(eigenvalue1 / total_variance, 3), "pc2": round(eigenvalue2 / total_variance, 3)})
print("Top PC1 loadings:", [(name, round(value, 3)) for name, value in pc1_loadings[:4]])
print("Top PC2 loadings:", [(name, round(value, 3)) for name, value in pc2_loadings[:4]])
print("Projected coordinates:")
for row in rows[:6]:
    print(row["spectrum_id"], row["object_class"], round(row["pc1"], 3), round(row["pc2"], 3))


In [ ]:
test_ids = {"B003", "B005", "M002", "M005", "E003", "E005"}
train_rows = [row for row in rows if row["spectrum_id"] not in test_ids]
test_rows = [row for row in rows if row["spectrum_id"] in test_ids]

def count_by_class(sample_rows):
    counts = {}
    for row in sample_rows:
        counts[row["object_class"]] = counts.get(row["object_class"], 0) + 1
    return counts

print("train size:", len(train_rows), "test size:", len(test_rows))
print("train class counts:", count_by_class(train_rows))
print("test ids:", [row["spectrum_id"] for row in test_rows])

class_centroids = {}
for object_class in sorted({row["object_class"] for row in train_rows}):
    projected_points = [[row["pc1"], row["pc2"]] for row in train_rows if row["object_class"] == object_class]
    class_centroids[object_class] = [sum(values[index] for values in projected_points) / len(projected_points) for index in range(2)]

def squared_distance(left_vector, right_vector):
    return sum((left - right) ** 2 for left, right in zip(left_vector, right_vector))

prediction_rows = []
for row in test_rows:
    predicted_class = min(class_centroids, key=lambda object_class: squared_distance([row["pc1"], row["pc2"]], class_centroids[object_class]))
    prediction_rows.append({
        "spectrum_id": row["spectrum_id"],
        "true_class": row["object_class"],
        "predicted_class": predicted_class,
        "quality_flag": row["quality_flag"],
        "snr": row["snr"],
        "pc1": row["pc1"],
        "pc2": row["pc2"],
    })

confusion = {}
for prediction_row in prediction_rows:
    confusion.setdefault(prediction_row["true_class"], {})
    confusion[prediction_row["true_class"]][prediction_row["predicted_class"]] = confusion[prediction_row["true_class"]].get(prediction_row["predicted_class"], 0) + 1

accuracy = sum(row["true_class"] == row["predicted_class"] for row in prediction_rows) / len(prediction_rows)
recall_by_class = {}
for object_class, counts in confusion.items():
    recall_by_class[object_class] = round(counts.get(object_class, 0) / sum(counts.values()), 3)

print("class centroids:", {key: [round(value, 3) for value in values] for key, values in class_centroids.items()})
print("test predictions:")
for row in prediction_rows:
    print(row["spectrum_id"], row["true_class"], row["quality_flag"], "->", row["predicted_class"], (round(row["pc1"], 3), round(row["pc2"], 3)))
print("test accuracy:", round(accuracy, 3))
print("confusion matrix:", confusion)
print("recall by class:", recall_by_class)


In [ ]:
misclassified_rows = [row for row in prediction_rows if row["true_class"] != row["predicted_class"]]
print("misclassified rows:", misclassified_rows)

def nearest_training_neighbors(target_row, candidate_rows, neighbor_count=4):
    distances = []
    for candidate_row in candidate_rows:
        distances.append((
            math.sqrt(squared_distance([target_row["pc1"], target_row["pc2"]], [candidate_row["pc1"], candidate_row["pc2"]])),
            candidate_row["spectrum_id"],
            candidate_row["object_class"],
            candidate_row["quality_flag"],
        ))
    distances.sort(key=lambda item: item[0])
    return distances[:neighbor_count]

target_row = next(row for row in rows if row["spectrum_id"] == "E005")
neighbors = nearest_training_neighbors(target_row, train_rows)
print("nearest training neighbors for E005:", [(round(distance, 3), spectrum_id, object_class, quality_flag) for distance, spectrum_id, object_class, quality_flag in neighbors])
print("E005 diagnostic indices:", {
    "blue_red_slope": round(target_row["blue_red_slope"], 3),
    "balmer_index": round(target_row["balmer_index"], 3),
    "halpha_excess": round(target_row["halpha_excess"], 3),
})
print("E005 normalized flux:", [round(target_row[flux_name + "_norm"], 3) for flux_name in flux_names])

for comparison_id in ["B004", "E004"]:
    comparison_row = next(row for row in rows if row["spectrum_id"] == comparison_id)
    print(
        comparison_id,
        comparison_row["object_class"],
        comparison_row["quality_flag"],
        {
            "blue_red_slope": round(comparison_row["blue_red_slope"], 3),
            "balmer_index": round(comparison_row["balmer_index"], 3),
            "halpha_excess": round(comparison_row["halpha_excess"], 3),
        },
    )

print("Interpretation:")
print("  E005 keeps a slightly blue continuum and only weak Halpha excess after pixel repair.")
print("  In PC space it sits between a low-SNR emission source (E004) and a Balmer-dominated star (B004).")
print("  This is exactly the kind of sample that deserves manual follow-up instead of blind trust in the label or the model.")


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print(f"Optional matplotlib plots skipped: {exc}")
else:
    class_colors = {
        "balmer_star": "tab:blue",
        "metal_rich_star": "tab:orange",
        "emission_source": "tab:green",
    }
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    for row in rows:
        marker = "x" if row["quality_flag"] != "clean" else "o"
        axes[0].scatter(row["pc1"], row["pc2"], color=class_colors[row["object_class"]], marker=marker, s=70)
        axes[0].text(row["pc1"] + 0.03, row["pc2"] + 0.03, row["spectrum_id"], fontsize=8)
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")
    axes[0].set_title("PCA projection of teaching spectra")

    wavelength_grid = [3900, 4100, 4300, 4861, 5175, 6563, 6800]
    for spectrum_id in ["B004", "E004", "E005"]:
        row = next(item for item in rows if item["spectrum_id"] == spectrum_id)
        axes[1].plot(wavelength_grid, [row[flux_name + "_norm"] for flux_name in flux_names], marker="o", label=f"{spectrum_id} ({row['object_class']})")
    axes[1].set_xlabel("wavelength [A]")
    axes[1].set_ylabel("median-normalized flux")
    axes[1].set_title("A misclassified spectrum and its neighbors")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
try:
    from sklearn.decomposition import PCA as SklearnPCA
    from sklearn.neighbors import KNeighborsClassifier
except ModuleNotFoundError as exc:
    print(f"Optional scikit-learn comparison skipped: {exc}")
else:
    train_matrix = [[row[feature_name] for feature_name in standardized_flux_names] for row in train_rows]
    test_matrix = [[row[feature_name] for feature_name in standardized_flux_names] for row in test_rows]
    pca = SklearnPCA(n_components=2)
    projected_train = pca.fit_transform(train_matrix)
    projected_test = pca.transform(test_matrix)
    classifier = KNeighborsClassifier(n_neighbors=3)
    classifier.fit(projected_train, [row["object_class"] for row in train_rows])
    predicted_labels = classifier.predict(projected_test)
    sklearn_accuracy = sum(predicted_label == row["object_class"] for predicted_label, row in zip(predicted_labels, test_rows)) / len(test_rows)
    print("sklearn PCA explained variance:", [round(value, 3) for value in pca.explained_variance_ratio_])
    print("sklearn 3-NN test accuracy:", round(sklearn_accuracy, 3))
    print("sklearn predictions:", list(zip([row["spectrum_id"] for row in test_rows], predicted_labels.tolist())))


这个案例最值得记住的一点是：

> 光谱分类的真正难点，往往不在把大多数样本分开，而在处理那些低信噪比、带缺陷、靠近边界的样本。

PCA 和一个简单分类器已经足够帮我们找到主结构；而真正让项目像科研而不是习题的，是后面的错误案例复查。 
